<div class="alert alert-info">
    <strong>导航</strong>
    <ul>
        <li>上一节： <a href="9_20_polarization_faraday_interpretation.ipynb">9.20 偏振与 Faraday 诊断</a></li>
        <li>下一节： <a href="9_x_further_reading_and_workflow.ipynb">9.x 延伸阅读与后续实践方向</a></li>
    </ul>
</div>


## 9.21 短间距、Mosaicking 与联合成像：从 Missing Flux 到通量恢复

第 9.10 节已经引入 missing flux、negative bowl 和 feather 的基本思想。本节把这个主题推进到更接近真实项目的层面：如何判断一个科学问题是否需要短间距，什么时候紧凑阵列已经足够，什么时候必须加入单碟或 total power 数据，feather 与 joint deconvolution 的前提分别是什么。

短间距问题的根源不是图像算法“不聪明”，而是测量本身缺少低空间频率。干涉阵测量的是天空亮度的 Fourier 分量；最短基线附近缺失时，大尺度结构对应的信息没有进入数据。CLEAN 可以在已有采样约束内选择一个图像模型，却不能从空白的 uv 中创造总通量。对于点源或远小于最大可恢复尺度的结构，这个限制不明显；对于弥散 H I、分子云、星系盘、超新星遗迹、银河系前景或 Sunyaev-Zel'dovich 信号，它常常直接决定科学结论。


### 9.21.1 最大可恢复尺度：公式只是第一道门槛

最大可恢复尺度常用近似

$$
\theta_{\rm MRS}\simeq \kappa\frac{\lambda}{b_{\rm min}}
$$

估计，其中 $b_{\rm min}$ 是有效最短基线，$\kappa$ 的量级常取 $0.6$ 左右，但具体数值依赖阵列几何、权重、flag、mosaic、deconvolution 和可接受的通量损失。这个公式的价值是提供量级判断：波长越长或最短基线越短，可恢复的大尺度越大；但它不应被理解为一道硬边界。接近 $\theta_{\rm MRS}$ 的结构会逐渐被衰减，而不是突然消失。

实际判断还要区分三个尺度。第一是合成波束，对应最高有效空间频率，决定角分辨率。第二是最大可恢复尺度，对应低空间频率采样，决定大尺度结构是否可测。第三是主波束或 mosaic 覆盖，对应视场和灵敏度分布。一个源可以被完整放在主波束内，同时仍然因为缺短间距而丢失通量；也可以通过 mosaic 覆盖很大区域，但若没有足够短基线或 total power，零 spacing 附近的信息仍然缺失。


![uv overlap and MRS](figures/practical_short_spacing_uv_overlap_mrs.png)

左图用 uv 平面示意普通连通阵列的中心空洞。这个空洞对应天空中的大尺度结构；空洞越大，越容易出现 missing flux 和 negative bowl。右图示意紧凑配置、扩展配置和单碟信息在 Fourier 空间中的互补关系。真正有用的组合不只是“数据越多越好”，还需要在重叠空间频率上检查通量尺度、权重和 beam 模型是否一致。


### 9.21.2 Missing flux 与 negative bowl：图像伪影背后的 Fourier 约束

缺短间距最直观的表现是积分通量偏低。对于一个大尺度发射区，干涉阵保留了边缘、团块和小尺度起伏，却无法测量平滑背景。成像时，为了让模型的 Fourier 变换在未采样低频之外仍能匹配已采样数据，扩展发射周围常出现负碗状结构。negative bowl 并不是简单的清洁残差，也不一定能通过更深 CLEAN 消除；它通常反映了图像模型在缺失低频约束下被迫平衡正负结构。

这个问题在谱线 cube 中尤其危险。若每个通道的缺失通量不同，moment 0 图会低估柱密度，moment 1 图可能被小尺度亮结构主导，PV 图中弥散低速气体会被削弱。对于连续谱弥散源，频谱指数图还可能因为不同频段的最短基线和 uv 覆盖不同而产生假谱指数结构。因此，missing flux 的判断必须和科学量绑定：目标是点源峰值、总通量、大尺度形态、柱密度、速度场，还是功率谱。不同科学量对短间距的敏感性不同。


![negative bowl](figures/practical_short_spacing_negative_bowl_flux.png)

图中从同一个真实天空出发，比较了只保留高空间频率、只保留低空间频率和 feather 式组合后的效果。干涉阵图像保留了小尺度结构，但弥散背景被滤掉，并在源周围产生负碗。单碟图像保留总功率和大尺度结构，却缺少高分辨细节。组合的目标是让低频和高频信息在各自可靠的尺度上发挥作用。


### 9.21.3 Feather：Fourier 空间中的后处理组合

Feather 的基本思想是在 Fourier 空间中把单碟图像的低空间频率和干涉图像的高空间频率相加。概念上可写为

$$
I_{\rm comb}=\mathcal{F}^{-1}\left[W_{\rm SD}(k)\,\mathcal{F}(I_{\rm SD})+W_{\rm INT}(k)\,\mathcal{F}(I_{\rm INT})\right],
$$

其中 $W_{\rm SD}$ 在低空间频率处较大，$W_{\rm INT}$ 在高空间频率处较大，并在过渡区平滑衔接。这个表达隐藏了许多实践前提：单碟图像必须有正确的 beam 信息和通量单位；干涉图像和单碟图像必须在同一坐标、像元、频率、通量尺度和主波束定义下比较；单碟图像需要处理 baseline、扫描条纹和边界；干涉图像应尽量是线性、可靠的 restored image 或合适的模型产品。

Feather 的优点是透明、快速、易诊断。它的风险也来自这种后处理性质：若两套数据的通量尺度在重叠空间频率上不一致，feather 会把不一致直接写入组合图像；若单碟 beam 错误，低频权重会把 beam 模型误差转成大尺度亮度误差；若干涉图像已经被 CLEAN mask 或主波束边缘噪声强烈影响，组合结果也会继承这些问题。


![feather weighting](figures/practical_short_spacing_feather_weighting_scale.png)

左图显示 feather 权重在空间频率上的互补关系。右图显示一个实用检查：在单碟和干涉阵都有效的 overlap 区域，Fourier 振幅应大体一致。若 overlap 中单碟振幅系统偏高或偏低，通常需要先检查通量尺度、beam 面积、单位转换、主波束校正和 regridding，而不是直接组合。


### 9.21.4 Mosaicking：视场、灵敏度和短间距的关系

Mosaicking 的直接目标是扩大视场并改善灵敏度均匀性。每个 pointing 的天空响应被主波束加权，多个 pointing 叠加后形成 mosaic 灵敏度图。pointing 间距过大时，灵敏度会出现明显起伏，主波束边缘噪声和反卷积误差会在拼接后变成空间结构。常见的 hexagonal mosaic 间距会取主波束 FWHM 的大约一半量级，但不同望远镜和频段的推荐值可能略有不同。

Mosaicking 也会改变低空间频率响应，因为主波束乘以天空在 Fourier 空间中对应卷积。多 pointing 可以在一定程度上改善接近最短基线的采样，但它不能替代零 spacing。对于很大尺度的总通量，仍然需要 total power 或单碟数据。换言之，mosaic 解决的是“天空覆盖和灵敏度采样”问题，短间距解决的是“低空间频率测量”问题；二者相关，但不能互相替代。


![mosaic spacing](figures/practical_mosaic_spacing_sensitivity.png)

图中比较了稀疏 pointing 和接近 Nyquist 采样的 mosaic 灵敏度。稀疏 spacing 会在 pointing 之间留下明显低灵敏度区域；较密 spacing 使灵敏度更平滑，也让 deconvolution 面对的噪声和主波束变化更可控。真实观测设计中，还需要把频率变化导致的主波束变化、目标形态、观测时间和校准开销一起考虑。


### 9.21.5 Joint deconvolution 与多配置合并

Feather 通常在成像之后组合图像，而 joint deconvolution 试图在同一个成像和反卷积问题中同时使用多配置、mosaic 和 total power 信息。它的优势是可以在模型求解阶段同时面对不同分辨率和不同空间频率约束，避免后处理组合的一些不连续。它的代价是对权重、beam、噪声、主波束模型和数据一致性更敏感。若权重不合理，高分辨数据可能压制低分辨数据；若 total power 噪声或 baseline 问题没有处理好，大尺度伪结构会进入模型。

多配置干涉数据合并也不是简单拼接 Measurement Set。紧凑配置和扩展配置在 uv overlap 区域应给出一致振幅；不同观测日期、频段、flag 策略和权重定义可能使 nominal weight 与实际噪声不一致。实践中常需要检查 visibility weights、按 scan 或 execution block 的 RMS、重叠 uv 振幅、成像后的残差和分辨率目标。对于谱线数据，还要确保通道网格、速度定义、连续谱扣除和主波束处理一致。


![combination workflow](figures/practical_short_spacing_combination_decision_workflow.png)

图中给出一个实践判断链。首先由科学角尺度和表面亮度灵敏度决定是否需要低空间频率；随后比较阵列 beam、最大可恢复尺度、uv density 和 mosaic 视场；若紧凑配置仍不足，需要考虑 total power 或单碟；最后在 feather 和 joint deconvolution 之间选择，并用 flux recovery、negative bowl、残差、overlap 区域和外部通量基准检验结果。


### 9.21.6 与真实软件工作流的对应

在 CASA 中，`feather`、`sdintimaging`、mosaic `tclean`、多配置合并和 total power 处理分别对应不同层级的问题。WSClean、MIRIAD、GILDAS、ASKAPsoft、ALMA pipeline 或其他工具也提供类似能力，但默认权重、主波束模型、单位和输出产品可能不同。工具选择本身不是科学保证；关键是组合前后的诊断是否支持目标物理量。

一个短间距或联合成像结果至少应说明：目标最大角尺度和所需表面亮度灵敏度；阵列配置、最短基线、估计 MRS 和 flag 后的实际 uv 覆盖；mosaic pointing 间距和灵敏度图；是否加入紧凑配置、ACA、单碟或 total power；单碟 beam、单位、baseline 和 regridding；干涉图像和单碟图像的通量尺度比较；feather 或 joint deconvolution 的权重与参数；组合前后总通量、负碗、残差和 overlap 区域的检查。没有这些信息，所谓“恢复了短间距”就很难被独立判断。


### 9.21.7 本节结论

短间距问题把观测设计、Fourier 采样、成像权重和科学测量紧密连在一起。最大可恢复尺度公式提供第一眼判断，但真实风险取决于源结构、uv 覆盖、mosaic 采样、通量尺度和目标科学量。Feather 是透明的 Fourier 后处理组合，适合做清晰诊断；joint deconvolution 更统一，但对权重和模型一致性要求更高。

因此，短间距处理不应被看成“最后加一张单碟图”的步骤，而应从观测设计阶段开始考虑。对弥散发射而言，能否测到低空间频率，往往与能否测到总通量、柱密度、气体质量和大尺度形态同等重要。
